In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from processing import *
from wavelengths import *
from reprojection import View
from datetime import datetime, timedelta

In [2]:
q_V = 299792458 / 6173.341
q_B = q_V * 0.231
tuning_constant = 3.513e-4
temperature_constant = 4.01225e-2
alpha = 1.327124e20

In [3]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/vlos/*vlos*.fits'))

In [4]:
dates = []
offsets = []
biases = []
temperatures = []
velocities = []
accelerations = []
distances = []
contposes = []
soops = []
centers = []

for file in files:

    with fits.open(file) as hdul:
        data = hdul[0].data
        header = hdul[0].header
        fg_data = hdul['PHI_FITS_FG_settings'].data
        fpa_data = hdul['PHI_FITS_FPA_settings'].data


    nx, ny = header['NAXIS2'], header['NAXIS1']
    xc, yc = header['CRPIX2'] - 1, header['CRPIX1'] - 1
    x0, y0 = header['PXBEG2'] - 1, header['PXBEG1'] - 1

    rsun = header['RSUN_ARC'] / header['CDELT1']

    #xi, yi = np.mgrid[:nx, :ny]
    #mask = (xi - xc) ** 2 + (yi - yc) ** 2 < (rsun * 0.2) ** 2
    soop = header['SOOPNAME']
    velocity = header['OBS_VR']
    distance = header['DSUN_AU']
    temperature = header['FGOV1PT1']
    contposn = header['CONTPOSN']
    contpos = header['CONTPOS'] - 1
    date = datetime.fromisoformat(header['DATE-OBS'])
    acceleration = (header['OBS_VW'] ** 2 + header['OBS_VN'] ** 2) / header['DSUN_OBS'] - alpha / header['DSUN_OBS'] ** 2

    wvs = read_wavelengths(header)
    wv0 = np.delete(wvs, contpos)[2]

    biases += [np.nanmedian(data) / q_V]
    data = (data + velocity) / q_V
    #biases += [np.nanmedian(data) + 6173.341 - wv0]

    data -= temperature_constant * (temperature - 61)
    data /= tuning_constant ## in Volts

    dates += [date]
    soops += [soop]
    offsets += [np.nanmedian(data)]
    temperatures += [temperature]
    velocities += [velocity / 1000]
    distances += [distance]
    accelerations += [acceleration]
    contposes += [contposn]
    centers += [(xc + x0, yc + y0)]


dates = np.array(dates)
soops = np.array(soops)
biases = np.array(biases)
offsets = np.array(offsets)
temperatures = np.array(temperatures)
velocities = np.array(velocities)
distances = np.array(distances)
accelerations = np.array(accelerations)
contposes = np.array(contposes)
centers = np.array(centers)

In [5]:
fig, ax = plt.subplots(figsize=(10,8))
ax1 = ax.twinx()

ax.plot(dates, centers[:,1], '.')
ax1.plot(dates, accelerations, color='tab:orange')

#ax.set_ylim(980,1020)
ax.set_ylim(1020,1060)

fig.tight_layout()

In [6]:
#np.savez('tuning_constant.npz', dates=dates, soops=soops, biases=biases, offsets=offsets,
#         temperatures=temperatures, velocities=velocities, distances=distances, accelerations=accelerations, contposes=contposes)

In [7]:
fig, ax = plt.subplots(figsize=(12,8))
ax1 = ax.twinx()

colors = ['tab:blue', 'tab:green', 'tab:red']
for temperature, color in zip([56, 61, 66], colors):
    t = np.where(np.abs(temperatures - temperature) < 2)
    ax.plot(dates[t], biases[t], '.', color=color, label=temperature)

#ax1.plot(dates, distances, '--', color='gray')
ax1.plot(dates, velocities, '--', color='gray')
#ax1.plot(dates, accelerations, '--', color='gray')

contpos_ = contposes[0]
date_ = dates[0]
for contpos, date in zip(contposes[1:], dates[1:]):
    if contpos != contpos_:
        ax.axvspan(date_, date, color='tab:' + contpos_, alpha=0.1)
        date_ = date
        contpos_ = contpos
ax.axvspan(date_, date, color='tab:' + contpos_, alpha=0.1)


ax.set_xlabel('Date')
ax.set_ylabel(r'Offset, $\AA$')
#ax1.set_ylabel('Distance, AU')
ax1.set_ylabel('Velocity, km/s')
#ax1.set_ylabel(r'Acceleration, m/s$^2$')

ax.set_xlim(dates[0], dates[-1])
ax.set_ylim(-0.03, 0)

ax.grid(True)
ax.legend()
fig.tight_layout()

In [8]:
np.mean(biases), np.std(biases)

(np.float32(-0.016295347), np.float32(0.0065787206))

In [9]:
def make_plot(offsets, velocities, temperatures, contposes, fit=True, fig=None, ax=None, **kwargs):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10,8))

    t_blue = np.where(contposes == 'blue')[0]
    ax.scatter(offsets[t_blue], velocities[t_blue], color='blue', **kwargs)

    t_red = np.where(contposes == 'red')[0]
    ax.scatter(offsets[t_red], velocities[t_red], color='red', **kwargs)


    t_56 = np.where(np.all([#offsets < 0,
                            np.abs(temperatures - 56) < 1], axis=0))[0]
    if (len(t_56) > 1) & fit:
        k_56, b_56 = np.polyfit(offsets[t_56], velocities[t_56], 1)
        plt.plot([850,-850], [k_56 * 850 + b_56, k_56 * -850 + b_56], '--', color='k', lw=0.5)
        print(k_56 / q_V * 1e3, b_56 / q_V * 1e3)


    t_61 = np.where(np.all([#offsets > 0,
                            contposes == 'blue',
                            np.abs(temperatures - 61) < 1], axis=0))[0]
    if (len(t_61) > 1) & fit:
        k_61, b_61 = np.polyfit(offsets[t_61], velocities[t_61], 1)
        plt.plot([-850,850], [k_61 * -850 + b_61, k_61 * 850 + b_61], '--', color='k', lw=0.5)
        print(k_61 / q_V * 1e3, b_61 / q_V * 1e3)

    t_61_ = np.where(np.all([contposes == 'red',
                             np.abs(temperatures - 61) < 1], axis=0))[0]

    if (len(t_61_) > 1) & fit:
        k_61_, b_61_ = np.polyfit(offsets[t_61_], velocities[t_61_], 1)
        plt.plot([-850,850], [k_61_ * -850 + b_61_, k_61_ * 850 + b_61_], '--', color='k', lw=0.5)
        print(k_61_ / q_V * 1e3, b_61_ / q_V * 1e3)


    t_66 = np.where(np.all([#offsets < 400,
                            np.abs(temperatures - 66) < 1], axis=0))[0]
    if (len(t_66) > 1) & fit:
        k_66, b_66 = np.polyfit(offsets[t_66], velocities[t_66], 1)
        plt.plot([-850,850], [k_66 * -850 + b_66, k_66 * 850 + b_66], '--', color='k', lw=0.5)
        print(k_66 / q_V * 1e3, b_66 / q_V * 1e3)


    #print(k_56 / q_V * 1e3, k_61 / q_V * 1e3, k_66 / q_V * 1e3)

    ax.set_xlabel('Offset, V')
    ax.set_ylabel('S/C velocity, km/s')
    #ax.legend()

    ax.set_xlim(-1000,1000)
    ax.set_ylim(-30,30)
    ax.grid(True)
    fig.tight_layout()

    return fig, ax

In [10]:
synoptics = np.any([soops == 'R_FULL_LRES_LCAD_RS-Synoptics-Low',
                    soops == 'R_FULL_LRES_LCAD_RS-Synoptics-High',
                    ], axis=0)

t0 = np.where(np.all([~synoptics,
                     dates > datetime(2026,1,1),
                     dates < datetime(2027,1,1)], axis=0))[0]

t1 = np.where(np.all([synoptics,
                     dates > datetime(2026,4,1),
                     dates < datetime(2027,1,1)], axis=0))[0]#[::4]



#fig, ax = make_plot(offsets[t0], velocities[t0], temperatures[t0], contposes[t0], label='non-synoptic', marker='x', s=20, fit=False)
fig, ax = make_plot(offsets[t1], velocities[t1], temperatures[t1], contposes[t1], marker='.', s=10)#, ax=ax, fig=fig)

0.0003515909489954431 -0.183772607719532
0.0003359016127454951 0.026150485498497245
0.0003860729386889889 0.027474910145879546
0.00032080777950221304 0.21750103086940484


In [11]:
def fit(T, contpos):
    k_56, b_56 = 0.0003515909489954431, -0.183772607719532
    k_61, b_61 = 0.0003359016127454951, 0.026150485498497245
    k_61_, b_61_ = 0.0003860729386889889, 0.027474910145879546
    k_66, b_66 = 0.00032080777950221304, 0.21750103086940484

    return ((np.abs(T - 56) < 1) * k_56 + (np.abs(T - 61) < 1) * (k_61 * (contpos == 'blue') + k_61_ * (contpos == 'red')) + (np.abs(T - 66) < 1) * k_66,
            (np.abs(T - 56) < 1) * b_56 + (np.abs(T - 61) < 1) * (b_61 * (contpos == 'blue') + b_61_ * (contpos == 'red')) + (np.abs(T - 66) < 1) * b_66)


In [12]:
k, b = fit(temperatures, contposes)
#k = 3.513e-4
#b = (temperatures - 61) * 4.01225e-2

biases_ = offsets * k + b - velocities * 1000 / q_V

In [13]:
np.mean(biases_), np.std(biases_)

(np.float64(-0.00033131024268446234), np.float64(0.005362971088040135))

In [14]:
fig, ax = plt.subplots(figsize=(12,8))
ax1 = ax.twinx()

colors = ['tab:blue', 'tab:green', 'tab:red']
for temperature, color in zip([56, 61, 66], colors):
    t = np.where(np.abs(temperatures - temperature) < 2)
    ax.plot(dates[t], biases_[t], '.', color=color, label=temperature)

#ax1.plot(dates, distances, '--', color='gray')
#ax1.plot(dates, velocities, '--', color='gray')
ax1.plot(dates, accelerations, '--', color='gray')
#ax1.plot(dates, centers[:,1], '--', color='gray')

contpos_ = contposes[0]
date_ = dates[0]
for contpos, date in zip(contposes[1:], dates[1:]):
    if contpos != contpos_:
        ax.axvspan(date_, date, color='tab:' + contpos_, alpha=0.1)
        date_ = date
        contpos_ = contpos
ax.axvspan(date_, date, color='tab:' + contpos_, alpha=0.1)


ax.set_xlabel('Date')
ax.set_ylabel(r'Offset, $\AA$')
#ax1.set_ylabel('Distance, AU')
#ax1.set_ylabel('Velocity, km/s')
#ax1.set_ylabel(r'Acceleration, m/s$^2$')

ax.set_xlim(dates[0], dates[-1])
#ax.set_ylim(-0.030, 0.0)
ax.set_ylim(-0.015, 0.015)
#ax1.set_ylim(980, 1020)
#ax1.set_ylim(1020, 1060)

ax.grid(True)
ax.legend()
fig.tight_layout()

(np.float64(-0.0008732880027657382), np.float64(0.0044622835883136765))

In [48]:
0.05 / 15 / 24 / 60

2.3148148148148148e-06